[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Mechanistic_Interpretability.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Mechanistic Interpretability

Open the model and find the *mechanism*: we train a tiny transformer on a task with a known algorithm (detecting balanced parentheses), then locate where the network computes what — attention maps, linear probes, and the causal test that separates correlation from mechanism: **activation patching**.

## 1. Pre-requisites

[Intro to Transformers](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb), [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb); [Causal Inference](./Causal_Inference.ipynb) supplies the intervention mindset.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# task with a KNOWN algorithm: is a ()-string balanced? ground truth = running-depth check
VOCAB = {"(": 0, ")": 1, "PAD": 2}
L_seq = 16
def make_batch(B):
    xs, ys, depths = [], [], []
    for _ in range(B):
        if rng.random() < 0.5:                                # balanced: random matched string
            s = []
            depth = 0
            for i in range(L_seq):
                if depth == 0 or (rng.random() < 0.5 and depth < L_seq - i - depth):
                    s.append("("); depth += 1
                else:
                    s.append(")"); depth -= 1
            if depth > 0: s[-depth:] = [")"]*depth
        else:                                                  # corrupt one position
            s = ["(", ")"]*(L_seq//2)
            s = list(rng.permutation(s))
        d = np.cumsum([1 if c == "(" else -1 for c in s])
        xs.append([VOCAB[c] for c in s])
        ys.append(int(d[-1] == 0 and d.min() >= 0))
        depths.append(d)
    return torch.tensor(xs), torch.tensor(ys), np.array(depths)

---
### 🕐 Session 1 of 2 — *Probes & Attention Maps* (~40 min)
**Goal:** train the model; find WHERE the running depth lives with linear probes.
**Builds on:** [Transformers workshop](../Intro_DL_4_Physics/intro_transformers/intro_transformers.ipynb). &nbsp; **Feeds into:** Session 2 (activation patching).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Probes & Attention Maps</b></summary>

**Timing (~40 min).** 8 min why a known-algorithm task · 10 min training · 12 min the probe · 10 min reading the R² honestly.

**Explain the experimental design first, because it is what makes interpretability teachable here.** Balanced parentheses has a **one-line algorithm**: track the running depth, check it never goes negative and ends at zero. So we know in advance what a correct solution must compute. **Interpretability on a real LLM is hard partly because nobody knows the right answer** — here we do, so every claim about the network can be checked against ground truth.

**State the hypothesis explicitly before probing.** *If the model solves this task, something inside it should represent the running depth.* That is a falsifiable prediction, and a linear probe — a least-squares fit from hidden states to the known depth — is the test. **Write the hypothesis on the board before running the cell**; a probe run without a prior prediction is fishing.

**Prepare the room to read $R^2 = 0.496$ as a partial success, not a discovery.** Half the variance of the running depth is linearly recoverable from layer 2. That is real evidence of a representation and it is **not** the tidy story the printed conclusion implies. Something depth-like is there; the model may also be using a nonlinear encoding, or a different sufficient statistic, or a mixture. **"Depth is represented" and "half of depth's variance is linearly decodable" are different claims**, and only the second is supported.

**Do not skip the layer-0 number, because it is the most instructive one.** The embeddings score $R^2 = 0.120$ — but embeddings only know the current token and its position. They **cannot** know the running depth. So where does 0.12 come from? **Position correlates with depth statistically**: early positions tend to have low depth, later ones higher. The probe is reading a confound. **A probe can succeed without the information being there**, which is exactly why Session 2's causal test exists.

**That connects directly to the [Causal Inference](./Causal_Inference.ipynb) workshop and is worth naming.** A probe measures *decodability*, which is an associational quantity. Whether the model *uses* the representation is an interventional question, and no amount of probing answers it. **Probes are the regression; patching is the do-operator.** Rooms that have done the causal workshop will recognise the structure immediately.

**Flag the 100% task accuracy as something to be suspicious of, not pleased by.** The two classes come from **different generators**: positives are constructed balanced strings, negatives are random permutations of eight of each token. A model could plausibly separate them on distributional quirks of the generators rather than by computing depth. **Ask the room how to rule that out** — the answer is a test set where both classes come from the *same* generator, which is a fifteen-line change and a genuinely better experiment.

**If time allows, run the probe per position rather than pooled.** Depth is meaningful at every position, and a per-position probe shows whether the representation is uniform or concentrated. That is a better picture of the mechanism than a single pooled number, and it sets up the patching heatmap directly.
</details>

## 2. The Model, and the Hypothesis

💡 **Intuition.** Balanced-parentheses has a one-line algorithm: track the running depth, check it never dips below zero and ends at zero. If our transformer learns the task, *something inside it should represent the running depth*. A **linear probe** — a tiny regression from hidden states to the known quantity — tests exactly that, layer by layer and position by position. Finding a probe that works is evidence of a representation; Session 2 tests whether the model actually *uses* it.

In [2]:
class TinyTf(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.emb = nn.Embedding(3, d)
        self.pos = nn.Embedding(L_seq, d)
        layer = nn.TransformerEncoderLayer(d, 4, 64, batch_first=True, norm_first=True, dropout=0)
        self.l1 = nn.TransformerEncoderLayer(d, 4, 64, batch_first=True, norm_first=True, dropout=0)
        self.l2 = nn.TransformerEncoderLayer(d, 4, 64, batch_first=True, norm_first=True, dropout=0)
        self.head = nn.Linear(d, 2)
    def forward(self, x, return_h=False):
        h0 = self.emb(x) + self.pos(torch.arange(L_seq))
        h1 = self.l1(h0)
        h2 = self.l2(h1)
        out = self.head(h2.mean(1))
        return (out, [h0, h1, h2]) if return_h else out

model = TinyTf()
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
for step in range(2500):
    xb, yb, _ = make_batch(256)
    loss = Fn.cross_entropy(model(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()
xb, yb, _ = make_batch(2000)
print(f"task accuracy: {(model(xb).argmax(1) == yb).float().mean():.1%}")

task accuracy: 100.0%


**What just happened.** A two-block transformer with 32-dimensional embeddings reached **100.0% accuracy** on 2,000 held-out strings after 2,500 steps. The task is solved.

**Which immediately raises the question this workshop exists to answer: solved *how*?** Balanced parentheses has a known algorithm — track the running depth, check it never goes negative and ends at zero. **A network at 100% must be computing something sufficient, but nothing so far says it computes that.** The rest of the notebook is the attempt to find out, and having a known correct algorithm is what makes the attempt checkable.

**Before celebrating the number, look at where the two classes come from, because they are generated differently.** Positives are *constructed* balanced strings, built by a loop that maintains depth. Negatives are random **permutations of eight `(` and eight `)`**. Those are different distributions, and a model could in principle separate them on generator artefacts — local run-length statistics, say — without ever computing a depth. **100% accuracy is consistent with the model having learned the algorithm and also with it having learned a shortcut.**

**The check is fifteen lines and worth running.** Build a test set where *both* classes come from the same generator: take constructed balanced strings and flip one character to make the negatives, which is exactly what `matched_pair` does in Session 2. If accuracy holds up on that harder set, the shortcut hypothesis is dead. **Distinguishing "learned the task" from "learned the dataset" is the first question to ask of any accuracy number**, and it is especially pressing when the number is 100%.

**Note the architecture choices that make interpretability feasible here.** Two blocks, $d = 32$, four heads, no dropout, and `norm_first` — small enough that every hidden state can be probed exhaustively and every layer patched. **Real interpretability work fights scale constantly**; this model is deliberately in the regime where exhaustive analysis is cheap.

**One structural detail that will matter in Session 2.** The head reads `h2.mean(1)` — a **mean-pool over all 16 positions**. So the verdict is an average of per-position contributions, and no single position can carry more than about $1/16$ of it by construction. That fact explains the patching numbers before you see them, and it is an architectural property rather than a discovery about the mechanism.

**Finally, keep the epistemic order straight.** The accuracy establishes that a sufficient computation exists inside this network. It says nothing about what that computation is, whether it resembles the human algorithm, or whether it generalises past the training distribution. **Everything after this cell is the work of turning "it works" into "here is how".**

In [3]:
# probe every layer for the RUNNING DEPTH (per position) — where does the algorithm live?
xb, yb, depths = make_batch(3000)
with torch.no_grad():
    _, hs = model(xb, return_h=True)
d_flat = torch.tensor(depths, dtype=torch.float32).reshape(-1)
print("linear-probe R² for running depth, by layer:")
for li, h in enumerate(hs):
    H = h.reshape(-1, h.shape[-1])
    H1 = torch.cat([H, torch.ones(len(H), 1)], 1)
    w = torch.linalg.lstsq(H1, d_flat[:, None]).solution
    pred = (H1 @ w).squeeze()
    r2 = 1 - ((pred - d_flat)**2).mean()/d_flat.var()
    print(f"  layer {li} ({'embeddings' if li==0 else f'after block {li}'}): R² = {r2:.3f}")
print("→ depth is not in the embeddings (they only know the current token) but EMERGES in the blocks —")
print("  attention is how position i gathers the count of what came before")

linear-probe R² for running depth, by layer:
  layer 0 (embeddings): R² = 0.120
  layer 1 (after block 1): R² = 0.431
  layer 2 (after block 2): R² = 0.496
→ depth is not in the embeddings (they only know the current token) but EMERGES in the blocks —
  attention is how position i gathers the count of what came before


**What just happened.** A linear probe for the running depth, layer by layer:

| layer | what it can see | $R^2$ |
|---|---|---|
| 0 — embeddings | current token + position only | **0.120** |
| 1 — after block 1 | + one round of attention | 0.431 |
| 2 — after block 2 | + two rounds | **0.496** |

**The rise from 0.12 to 0.50 is the finding, and its mechanism is attention.** An embedding knows one token; the running depth is a **cumulative** quantity requiring information from every earlier position. Attention is the only operation in the architecture that moves information across positions, so depth-like structure could not appear before the blocks and does appear after them. **The layer index is doing causal work here**, which is what makes the trend more informative than any single number.

**But the layer-0 value is the one to stop on, because it should be zero and is not.** Embeddings genuinely cannot know the running depth — they see one token and one position. So where does $R^2 = 0.12$ come from? **Position is correlated with depth**: early positions have small depths, later ones larger, purely as a statistical fact about the string distribution. The probe is reading a **confound**, not a representation.

**That is the central methodological warning of the session, and it generalises far beyond this demo.** A probe measures whether a quantity is **decodable**, and decodability can come from correlation, from a confound, or from the probe itself doing the computation. Probe a large enough hidden state with a flexible enough probe and you can decode almost anything — including quantities the model demonstrably does not use. **Finding a probe that works is evidence a representation *might* be there. It is not evidence the model uses it.**

**And be careful with the 0.496 too, since the printed conclusion overstates it.** Half the variance of the running depth is *linearly* recoverable from layer 2. The honest claim is exactly that — not "the model represents depth". The other half may be encoded nonlinearly, or the model may be tracking a different sufficient statistic that happens to correlate with depth at $R^2 = 0.5$. **"Depth is represented" and "half of depth's variance is linearly decodable" are different sentences**, and only the second is supported by this cell.

**A cheaper, sharper version of this experiment is worth suggesting.** Probe *per position* rather than pooling all $3000 \times 16$ states together. Depth is a per-position quantity, and pooling hides whether the representation is uniform across the sequence or concentrated near the end where the verdict is decided. A per-position $R^2$ curve is a much better picture of the mechanism and costs one extra loop.

**Finally, note exactly what question remains open.** This is the associational half of the investigation — the regression, in the language of [Causal Inference](./Causal_Inference.ipynb). Whether the model *uses* what the probe found is an **interventional** question, and no probe of any sophistication answers it. **Session 2 is the do-operator**, and it is the reason this workshop does not stop here.

---
### 🕐 Session 2 of 2 — *Activation Patching: the Causal Test* (~40 min)
**Goal:** swap internal activations between clean and corrupted runs — which components MATTER?
**Builds on:** Session 1; [Causal Inference](./Causal_Inference.ipynb).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Activation Patching — the Causal Test</b></summary>

**Timing (~40 min).** 10 min why probes are not enough · 10 min the patching procedure · 12 min the results · 8 min what the full-layer number does *not* show.

**Open by naming the gap Session 1 left.** A probe found depth-like structure. That establishes the information is **present**, not that it is **used**. This is precisely the associational/interventional distinction from [Causal Inference](./Causal_Inference.ipynb), transplanted inside a network: probing is the regression, patching is the do-operator. **You cannot settle a usage question with a decoding experiment**, at any level of probe sophistication.

**Describe the intervention in one sentence before any code.** Run a balanced string and a corrupted one; **transplant** an activation from the clean run into the corrupted run; if the verdict moves toward "balanced", that activation causally carries the verdict. Normalise by the clean/corrupted logit gap so the answer is a fraction between 0 and 1. **That is Pearl's $do(\cdot)$ with a forward hook.**

**Emphasise the design of `matched_pair`, because it is what makes the comparison clean.** Clean and corrupted differ by **exactly one flipped character**. Everything else — length, position embeddings, most of the token sequence — is held fixed. **A minimal pair isolates the effect**, in exactly the way a controlled experiment does, and building good minimal pairs is most of the craft in real patching studies.

**Now the result the room must not misread: full-layer patch recovery is +1.00 for *both* blocks.** Ask why before explaining. The answer is that it is **near-tautological**. Patching all of block 1's output replaces the entire residual stream; block 2 then recomputes from clean input, so the model produces the clean verdict. **A full-layer patch at any layer recovers ~1.00 in any network** — it is a correctness check on the patching harness, not a finding about the mechanism. Presenting it as evidence of localisation would be wrong.

**Then the number that does carry information: mean single-position recovery of +0.05 and +0.06.** Sixteen positions at roughly 0.05 sum to about 0.8 — so the verdict is **distributed nearly uniformly across positions**, not concentrated in a few. Given that the head is `h2.mean(1)`, a mean-pool over 16 positions, that is close to what the architecture forces. **The honest conclusion is "distributed and roughly as diluted as the pooling predicts", not "a localised circuit".** The notebook's closing line overstates this and the debrief corrects it.

**Which makes the heatmap the interesting artefact rather than the summary numbers.** Look for positions that exceed the 0.05 baseline — near the flip site, or near the end of the string where the final-depth check must resolve. **Deviation from uniform is the signal**; the mean is a null result by construction.

**Give the room the experiment that would actually localise the mechanism.** Patch **relative to the flip position** rather than absolute position: average recovery at flip$-2$, flip$-1$, flip, flip$+1$, and so on. Since the flip site varies between pairs, absolute-position averaging smears exactly the structure you are looking for. **That one change is the difference between a smeared heatmap and a readable one**, and it is a fifteen-line edit.

**Close by placing the method honestly.** Activation patching is the core technique of modern interpretability research — it is how induction heads and indirect-object-identification circuits were found in real models. It is also **expensive** (one forward pass per patch site), sensitive to the choice of corruption, and gives no guarantee that a recovered circuit is the only one. **Scaling it to frontier models is an open problem and an actively hiring research field**, which is a fair and motivating note to end a workshop on.
</details>

## 3. From Correlation to Mechanism

💡 **Intuition.** A probe finding depth proves the information is *present*, not that it's *used* — the [collider lesson](./Causal_Inference.ipynb) for neural nets. **Activation patching** is the intervention: run a balanced string and an unbalanced one; copy one layer's activations from the balanced run into the unbalanced run; if the verdict flips toward 'balanced', that layer *causally carries* the verdict. Do it per layer and position and you map the circuit. This do-operator-for-networks is the core method of modern interpretability research.

In [4]:
# clean = balanced string; corrupted = same string with ONE paren flipped
def matched_pair():
    while True:
        xb, yb, _ = make_batch(64)
        for i in range(64):
            if yb[i] == 1:
                x_clean = xb[i]
                flip = rng.integers(2, L_seq-2)
                x_corr = x_clean.clone(); x_corr[flip] = 1 - x_corr[flip]
                d = np.cumsum([1 if c == 0 else -1 for c in x_corr])
                if not (d[-1] == 0 and d.min() >= 0):
                    return x_clean[None], x_corr[None], flip

def patched_logit_gain(layer_idx, pos, x_clean, x_corr):
    """run corrupted input, but transplant one clean activation; return balanced-logit recovery"""
    acts = {}
    def hook_store(mod, inp, out): acts["clean"] = out.detach().clone()
    def hook_patch(mod, inp, out):
        out = out.clone()
        if pos is None: out[:] = acts["clean"]                # full-layer patch
        else: out[:, pos] = acts["clean"][:, pos]
        return out
    layer = [model.l1, model.l2][layer_idx]
    h = layer.register_forward_hook(hook_store)
    with torch.no_grad(): base_clean = model(x_clean)
    h.remove()
    with torch.no_grad(): base_corr = model(x_corr)
    h = layer.register_forward_hook(hook_patch)
    with torch.no_grad(): patched = model(x_corr)
    h.remove()
    span = (base_clean[0,1]-base_clean[0,0]) - (base_corr[0,1]-base_corr[0,0])
    gain = (patched[0,1]-patched[0,0]) - (base_corr[0,1]-base_corr[0,0])
    return float(gain/span) if abs(span) > 1e-6 else 0.0

heat = np.zeros((2, L_seq))
n_pairs = 40
for _ in range(n_pairs):
    x_clean, x_corr, flip = matched_pair()
    for li in range(2):
        for p in range(L_seq):
            heat[li, p] += patched_logit_gain(li, p, x_clean, x_corr)/n_pairs

plt.figure(figsize=(8.5, 2.4))
plt.imshow(heat, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="verdict recovery")
plt.yticks([0, 1], ["after block 1", "after block 2"]); plt.xlabel("patched position")
plt.title("activation patching: which (layer, position) causally carries the verdict")
plt.tight_layout(); plt.show()
# single-position patches are diluted by the 16-position mean-pool — patch WHOLE layers too
full = np.zeros(2)
for _ in range(n_pairs):
    x_clean, x_corr, flip = matched_pair()
    for li in range(2):
        full[li] += patched_logit_gain(li, None, x_clean, x_corr)/n_pairs
print(f"FULL-layer patch recovery:  block 1 {full[0]:+.2f}   block 2 {full[1]:+.2f}")
print(f"mean SINGLE-position recovery: block 1 {heat[0].mean():+.2f}, block 2 {heat[1].mean():+.2f}")
print("→ transplanting a whole layer's activations transfers the verdict almost completely;")
print("  single positions carry only slivers (the verdict is mean-pooled) — the heatmap shows")
print("  WHICH slivers matter: a localized circuit, causally mapped")

FULL-layer patch recovery:  block 1 +1.00   block 2 +1.00
mean SINGLE-position recovery: block 1 +0.05, block 2 +0.06
→ transplanting a whole layer's activations transfers the verdict almost completely;
  single positions carry only slivers (the verdict is mean-pooled) — the heatmap shows
  WHICH slivers matter: a localized circuit, causally mapped


/tmp/ipykernel_288551/1243443074.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Two very different measurements, and only one of them means anything:

| patch | block 1 | block 2 |
|---|---|---|
| whole layer | **+1.00** | **+1.00** |
| mean single position | +0.05 | +0.06 |

**Start with the +1.00, because it is close to tautological and should not be read as a finding.** Patching *all* of block 1's output replaces the **entire residual stream**; block 2 then recomputes from clean input and the head sees a clean representation, so of course the clean verdict comes out. **A full-layer patch recovers ~1.00 at any layer of any network** — it confirms the hooks are wired correctly and nothing more. It is a harness test, not evidence that either block is where the verdict lives.

**The informative number is the single-position mean, and it says the mechanism is *distributed*.** About 0.05 per position, sixteen positions, summing to roughly 0.8 — close to full recovery spread nearly evenly. **No individual position carries the verdict.** Given that the head is `h2.mean(1)`, a mean-pool over all 16 positions, that is very close to what the architecture forces: each position can contribute at most about $1/16 = 0.0625$ of the pooled output.

**So the printed conclusion — "a localized circuit, causally mapped" — is stronger than the data supports.** The measured pattern is "distributed, and about as diluted as the mean-pool predicts". **A uniform result is a null result**, and calling it localisation reads structure into a flat line. The genuinely interesting question is whether any position *exceeds* the 0.05 baseline, which is what the heatmap is for and what the summary numbers average away.

**And there is a design flaw making the heatmap harder to read than it needs to be.** The flip position is **random for every pair** (`rng.integers(2, L_seq-2)`), but recovery is averaged over **absolute** position. So whatever structure exists around the corruption site is smeared across all positions by the averaging. **Align to the flip instead** — record recovery at flip$-2$, flip$-1$, flip, flip$+1$, … and average those. That single change is the difference between a flat heatmap and a readable one, and it is about fifteen lines.

**What the cell does establish, and it is worth stating clearly, is the method.** Run a clean input and a corrupted one differing by **exactly one character**; transplant an internal activation from clean into corrupted; measure how far the verdict moves, normalised by the clean/corrupted gap. **That is Pearl's $do(\cdot)$ implemented with a forward hook** — an intervention, not an observation, and therefore capable of answering the question a probe cannot.

**Which closes the loop the workshop opened.** Session 1's probe found depth-like structure at $R^2 = 0.50$ and could not say whether the model uses it — the same limitation the [Causal Inference](./Causal_Inference.ipynb) workshop identified for regression on observational data. Patching is the intervention. **The result here is that the verdict is carried diffusely rather than by an identifiable component**, which is a real answer, just not a tidy one.

**Finally, the honest note about the field.** Activation patching is how induction heads and the indirect-object-identification circuit were found in real models — it works. It is also expensive (one forward pass per patch site), sensitive to how you choose the corruption, and silent about whether a recovered circuit is the only one. **Scaling it is an open problem**, which is why this is an actively hiring research area rather than a solved technique.

## 4. Conclusion

Probes locate representations (depth R² rising through the blocks); patching tests *use* (verdict recovery mapped by layer and position). Correlation-to-causation, inside the network — the same discipline [Causal Inference](./Causal_Inference.ipynb) taught for the world outside. Scaling these methods to frontier models is an open, hiring-hot research field.

---
## Where next

- [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb) — apply both tools to the fable nano-GPT you trained there.
- [Causal Inference](./Causal_Inference.ipynb) — the intervention logic, formalized.